# Attribute Information:
1. drugName (categorical): name of drug
2. condition (categorical): name of condition
3. review (text): patient review
4. rating (numerical): 10 star patient rating
5. date (date): date of review entry
6. usefulCount (numerical): number of users who found review useful

In [1]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\abdur\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [2]:
train_data = pd.read_csv(r'C:\Self Learning\Pytorch\RNN\Drug Rating Prediction from review RNN\drug_review_test.csv')
test_data = pd.read_csv(r'C:\Self Learning\Pytorch\RNN\Drug Rating Prediction from review RNN\drug_review_train.csv')
val_data = pd.read_csv(r'C:\Self Learning\Pytorch\RNN\Drug Rating Prediction from review RNN\drug_review_validation.csv')

In [3]:
len(train_data) , len(test_data) , len(val_data)

(46108, 110811, 27703)

In [4]:
print(train_data.columns)
print(test_data.columns)
print(val_data.columns)

Index(['Unnamed: 0', 'patient_id', 'drugName', 'condition', 'review', 'rating',
       'date', 'usefulCount', 'review_length'],
      dtype='object')
Index(['Unnamed: 0', 'patient_id', 'drugName', 'condition', 'review', 'rating',
       'date', 'usefulCount', 'review_length'],
      dtype='object')
Index(['Unnamed: 0', 'patient_id', 'drugName', 'condition', 'review', 'rating',
       'date', 'usefulCount', 'review_length'],
      dtype='object')


In [5]:
df = pd.concat([train_data , test_data , val_data],axis=0)

In [6]:
print("size after merging ",len(df))
print("columns : ",df.columns)

size after merging  184622
columns :  Index(['Unnamed: 0', 'patient_id', 'drugName', 'condition', 'review', 'rating',
       'date', 'usefulCount', 'review_length'],
      dtype='object')


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 184622 entries, 0 to 27702
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Unnamed: 0     184622 non-null  int64  
 1   patient_id     184622 non-null  int64  
 2   drugName       184622 non-null  object 
 3   condition      184622 non-null  object 
 4   review         184622 non-null  object 
 5   rating         184622 non-null  float64
 6   date           184622 non-null  object 
 7   usefulCount    184622 non-null  int64  
 8   review_length  184622 non-null  int64  
dtypes: float64(1), int64(4), object(4)
memory usage: 14.1+ MB


In [8]:
df.drop(columns=['Unnamed: 0', 'patient_id', 'drugName', 'condition',
       'date', 'usefulCount', 'review_length'], inplace=True)
df.drop_duplicates(inplace=True)

In [9]:
df.head()

,review,rating
0,"""i've tried a few antidepressants over the yea...",10.0
1,"""my son has crohn's disease and has done very ...",8.0
2,"""contrave combines drugs that were used for al...",9.0
3,"""i have been on this birth control for one cyc...",9.0
4,"""4 days in on first 2 weeks. using on arms an...",4.0


In [ ]:
import re
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    tokens = text.split()
    return [stemmer.stem(w) for w in tokens if w not in stop_words]

In [11]:
tokenized = df["review"].apply(tokenize)

In [12]:
rows = []
for row in iter(tokenized):
    rows.append(row)

In [13]:
for row in rows[:10]:
    print(row)

['ive', 'tri', 'antidepress', 'year', 'citalopram', 'fluoxetin', 'amitriptylin', 'none', 'help', 'depress', 'insomnia', 'anxieti', 'doctor', 'suggest', 'chang', 'onto', 'mg', 'mirtazapin', 'medicin', 'save', 'life', 'thank', 'side', 'effect', 'especi', 'common', 'weight', 'gain', 'ive', 'actual', 'lost', 'alot', 'weight', 'still', 'suicid', 'thought', 'mirtazapin', 'save']
['son', 'crohn', 'diseas', 'done', 'well', 'asacol', 'complaint', 'show', 'side', 'effect', 'taken', 'mani', 'nine', 'tablet', 'per', 'day', 'one', 'time', 'ive', 'happi', 'result', 'reduc', 'bout', 'diarrhea', 'drastic']
['contrav', 'combin', 'drug', 'use', 'alcohol', 'smoke', 'opioid', 'cessat', 'peopl', 'lose', 'weight', 'also', 'help', 'control', 'over', 'doubt', 'obes', 'caus', 'sugarcarb', 'addict', 'power', 'drug', 'take', 'five', 'day', 'good', 'news', 'seem', 'go', 'work', 'immedi', 'feel', 'hungri', 'want', 'food', 'realli', 'dont', 'care', 'eat', 'fill', 'stomach', 'sinc', 'day', 'dont', 'know', 'ive', 'lo

In [16]:
# create vocab 
vocab = {"<PAD>":0 , "<UNK>":1}
def create_vocab(rows):
    for row in rows:
        for word in row:
            if word not in vocab:
                vocab[word] = len(vocab)
create_vocab(rows)
    

In [17]:
print("Size of vocab: ",len(vocab))
print(vocab)

Size of vocab:  55522
{'<PAD>': 0, '<UNK>': 1, 'ive': 2, 'tri': 3, 'antidepress': 4, 'year': 5, 'citalopram': 6, 'fluoxetin': 7, 'amitriptylin': 8, 'none': 9, 'help': 10, 'depress': 11, 'insomnia': 12, 'anxieti': 13, 'doctor': 14, 'suggest': 15, 'chang': 16, 'onto': 17, 'mg': 18, 'mirtazapin': 19, 'medicin': 20, 'save': 21, 'life': 22, 'thank': 23, 'side': 24, 'effect': 25, 'especi': 26, 'common': 27, 'weight': 28, 'gain': 29, 'actual': 30, 'lost': 31, 'alot': 32, 'still': 33, 'suicid': 34, 'thought': 35, 'son': 36, 'crohn': 37, 'diseas': 38, 'done': 39, 'well': 40, 'asacol': 41, 'complaint': 42, 'show': 43, 'taken': 44, 'mani': 45, 'nine': 46, 'tablet': 47, 'per': 48, 'day': 49, 'one': 50, 'time': 51, 'happi': 52, 'result': 53, 'reduc': 54, 'bout': 55, 'diarrhea': 56, 'drastic': 57, 'contrav': 58, 'combin': 59, 'drug': 60, 'use': 61, 'alcohol': 62, 'smoke': 63, 'opioid': 64, 'cessat': 65, 'peopl': 66, 'lose': 67, 'also': 68, 'control': 69, 'over': 70, 'doubt': 71, 'obes': 72, 'caus': 

In [26]:
def word_to_indices(rows , vocab):
    indices_rows = []
    for row in rows:
        indices = []
        for word in row:
            if word in vocab:
                indices.append(vocab[word])
            else:
                indices.append(vocab["<UNK>"])
        indices_rows.append(indices)
    return indices_rows
reviews = word_to_indices(rows , vocab)
labels = list(df.rating)

In [27]:
for idx in reviews[:10]:
    print(idx)

[2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 2, 30, 31, 32, 28, 33, 34, 35, 19, 21]
[36, 37, 38, 39, 40, 41, 42, 43, 24, 25, 44, 45, 46, 47, 48, 49, 50, 51, 2, 52, 53, 54, 55, 56, 57]
[58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 28, 68, 10, 69, 70, 71, 72, 73, 74, 75, 76, 60, 77, 78, 49, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 49, 90, 96, 2, 31, 28, 90, 97, 98, 85, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 58]
[114, 69, 50, 115, 116, 117, 118, 119, 114, 69, 120, 121, 122, 104, 123, 114, 69, 124, 125, 126, 127, 114, 69, 24, 25, 128, 129, 104, 130, 114, 69, 131, 132, 133, 134, 2, 85, 135, 136, 137, 138, 139, 49, 140, 141, 142, 143, 144, 145, 137, 146, 147, 148, 149, 150, 151, 106, 152, 153, 154, 155, 104, 156, 157, 158]
[49, 151, 159, 61, 160, 161, 162, 163, 164, 165, 166, 167, 168, 126, 169, 96, 45, 170, 171, 35, 172, 146, 129, 173, 174, 126, 79, 101, 143, 51, 175]
[2, 176

# Main dataloader and Dataset

In [55]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader , Dataset , random_split
from torch.nn.utils.rnn import pad_sequence , pack_padded_sequence
import matplotlib.pyplot as plt
from tqdm import tqdm
import numpy as np

In [30]:
reviews_tensor = [torch.tensor(review) for review in reviews]
labels_tensor = torch.tensor(labels)

In [ ]:
# Applying padding
padded_review_tensor = pad_sequence(reviews_tensor , batch_first=True , padding_value=0)

In [34]:
padded_review_tensor.shape , labels_tensor.shape

(torch.Size([110912, 950]), torch.Size([110912]))

In [68]:
class DrugReviewDataset(Dataset):
    def __init__(self , x , y):
        self.x = x
        self.y = y
    def __len__(self):
        return self.x.shape[0]
    def __getitem__(self, index):
        return self.x[index] , (self.y[index] - 1).to(torch.long)

In [69]:
# Divinding Into Batches
data = DrugReviewDataset(padded_review_tensor , labels_tensor)

splited_size = int(0.8 * len(data))
test_size = len(data) - splited_size

splited_data, test_data = random_split(data, [splited_size, test_size])

train_size = int(0.8 * len(splited_data))
val_size = len(splited_data) - train_size

train_data , val_data = random_split(splited_data , [train_size , val_size])

In [70]:
len(train_data) , len(test_data) , len(val_data)

(70983, 22183, 17746)

In [ ]:
class RNN_Review(nn.Module):
    def __init__(self , vocab_size , embedding_dim, rneuron_num, pad_idx):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size , embedding_dim , padding_idx=pad_idx)
        self.rnn = nn.RNN(embedding_dim , rneuron_num , batch_first=True)

        self.bnR = nn.BatchNorm1d(rneuron_num)
        self.dropout = nn.Dropout(0.3)

        self.fc4 = nn.Linear(rneuron_num , 10)

    def forward(self , x):
        actual_length = (x !=0).sum(dim=1)
        em = self.embedding(x)
        packed = pack_padded_sequence(em , actual_length.cpu() , batch_first=True , enforce_sorted=False) # For ignoring the paddings
        out , hidden = self.rnn(packed)
        hidden = self.bnR(hidden.squeeze_(0))
        hidden = self.dropout(hidden)
        fc = self.fc4(hidden)
        return fc

class Main_Exec():
    def __init__(self ,vocab , embedding_dim , rneuron_num, pad_idx , batch_size , lr , epochs, wdecay , step_size , gamma , optimizer):
        self.train_batches = DataLoader(train_data, batch_size=batch_size, shuffle=True)
        self.val_batches = DataLoader(val_data, batch_size=batch_size, shuffle=False)
        self.test_batches = DataLoader(test_data, batch_size=batch_size, shuffle=False)

        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = RNN_Review(len(vocab) , embedding_dim, rneuron_num , 0).to(self.device)
        if optimizer == "RMSprop":
            self.optimizer = optim.RMSprop(self.model.parameters() , lr=lr , weight_decay=wdecay)
        elif optimizer == "SGD":
            self.optimizer = optim.SGD(self.model.parameters() , lr=lr , weight_decay=wdecay)
        else:
            self.optimizer = optim.Adam(self.model.parameters() , lr=lr, weight_decay=wdecay)
        self.criterion = nn.CrossEntropyLoss()
        self.scheduler = optim.lr_scheduler.StepLR(self.optimizer, step_size=step_size, gamma=gamma)
        self.num_epochs = epochs

        self.train_loss_lis , self.val_loss_lis , self.train_acc_lis , self.val_acc_lis = [] , [] , [] , []

    def get_metrics(self):
        return self.train_loss_lis , self.val_loss_lis , self.train_acc_lis , self.val_acc_lis

    def calculate_accuracy(self, y_pred , y):
        y_pred = torch.argmax(y_pred , dim=1)
        accurate = (y_pred == y).sum().item()
        accuracy = accurate / len(y)
        return accuracy

    def test_model(self):
        self.model.eval()
        test_acc = 0.0
        with torch.no_grad():
            test_bar = tqdm(self.test_batches , desc=f"Test : ",leave=False)
            for x, y in test_bar:
                x , y = x.to(self.device) , y.to(self.device)
                y_pred = self.model(x)

                test_acc += self.calculate_accuracy(y_pred , y)
        return test_acc / len(self.test_batches)

    def train(self):
        self.train_loss_lis = []
        self.val_loss_lis = []
        self.train_acc_lis = []
        self.val_acc_lis = []
        abs_loss_deff = 0
        abs_acc_deff = 0
        for epoch in range(self.num_epochs):
            train_loss = 0.0
            val_loss = 0.0
            train_acc = 0.0
            val_acc = 0.0
            curr_loss_avg = 0
            curr_acc_avg = 0
            self.model.train()
            train_bar = tqdm(self.train_batches , desc=f"Training {epoch+1}/{self.num_epochs}",leave=False)
            for x, y in train_bar:
                x , y = x.to(self.device) , y.to(self.device)
                y_pred = self.model(x)
                loss = self.criterion(y_pred , y)
                self.optimizer.zero_grad()
                loss.backward()
                self.optimizer.step()
                train_acc += self.calculate_accuracy(y_pred , y)
                train_loss += loss.item()
            train_loss /= len(self.train_batches)
            train_acc /= len(self.train_batches)
            self.model.eval()
            with torch.no_grad():
                val_bar = tqdm(self.val_batches , desc=f"Validation : {epoch+1}/{self.num_epochs}",leave=False)
                for x, y in val_bar:
                    x , y = x.to(self.device) , y.to(self.device)
                    y_pred = self.model(x)
                    loss = self.criterion(y_pred , y)
                    val_loss += loss.item()
                    val_acc += self.calculate_accuracy(y_pred , y)
                val_acc /= len(self.val_batches)
                val_loss /= len(self.val_batches)
            self.scheduler.step()
            abs_loss_deff += abs(train_loss - val_loss)
            abs_acc_deff += abs(train_acc - val_acc)

            # MEtrics to check overfit
            curr_loss_avg = abs_loss_deff / (epoch+1)
            curr_acc_avg = abs_acc_deff / (epoch+1)

            self.train_loss_lis.append(train_loss)
            self.val_loss_lis.append(val_loss)
            self.train_acc_lis.append(train_acc)
            self.val_acc_lis.append(val_acc)
            print(f"Epoch {epoch+1}/{self.num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Train Acc: {train_acc:.4f} | val Acc: {val_acc:.4f} | Curr avg loss def {curr_loss_avg} | Curr avg acc def {curr_acc_avg}")

        avg_loss_deff , avg_acc_deff = (abs_loss_deff / self.num_epochs) , (abs_acc_deff / self.num_epochs)
        return avg_loss_deff , avg_acc_deff


    
    def plot_graphs(self):
        x = np.arange(len(self.train_loss))
        plt.figure(figsize=(20,8))
        plt.subplot(1,2,1)
        plt.plot(x , self.train_loss , label="Train Loss" , color="orange" ,linestyle="-.")
        plt.plot(x , self.val_loss , label="Val Loss" , color="red",linestyle="--")
        plt.title("Loss over epoch")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()

        plt.subplot(1,2,2)
        plt.plot(x , self.train_acc , label="Train Acc" , color="orange" ,linestyle="-.")
        plt.plot(x , self.val_acc , label="Val Acc" , color="red",linestyle="--")
        plt.title("Accuracy over epoch")
        plt.xlabel("Epoch")
        plt.ylabel("Accuracy")
        plt.legend()
        plt.show()





In [ ]:
# train_batches = DataLoader(train_data, batch_size=10, shuffle=True)
# x , y = next(iter(train_batches))
# model = RNN_Review(len(vocab) , 100 , 128 , 0)
# y_pred = model(x)
# loss = nn.CrossEntropyLoss()
# print(y_pred.shape , y.shape)
# loss(y_pred , y.to(torch.long))

torch.Size([10, 10]) torch.Size([10])


tensor(2.2639, grad_fn=<NllLossBackward0>)

In [73]:
# (vocab , embedding_dim , rneuron_num, pad_idx , batch_size , lr , epochs, wdecay , step_size , gamma , optimizer)
embedding_dim = 256
lr = 0.0001
batch_size = 512
pad_idx = 0
rneuron_num = 192
wdecay = 1e-5
epochs = 50 
step_size = 5
gamma = 0.1
optimizer = "RMSprop"

model = Main_Exec(vocab , embedding_dim , rneuron_num, pad_idx , batch_size , lr , epochs, wdecay , step_size , gamma , optimizer)
avg_loss_def , avg_acc_def = model.train()
testacc = model.test_model()
print("Accuracy : ", testacc)
print("--"*40)
model.plot_graphs()

Epoch 1/50 | Train Loss: 2.3421 | Val Loss: 2.2253 | Train Acc: 0.1555 | val Acc: 0.2161 | Curr avg loss def 0.11684581079316425 | Curr avg acc def 0.06059164802746045


KeyboardInterrupt: 